# Beautiful Soup 2 — Scrape the S&P 500 companies table

**Goal:** use the supplied Wikipedia *List of S&P 500 companies* page to repeat the Beautiful Soup fundamentals, then extract its constituents table into pandas.

## 1. Fetch the page

Import the request, parser, and tabular-data libraries.

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [3]:
WIKIPEDIA_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

### `requests.get` or `session.get`?

| | `requests.get(url, headers=HEADERS)` | `session.get(url)` |
|---|---|---|
| headers | passed **every** time | set **once** |
| cookies | forgotten after each call | **kept** for the next calls |
| connection | a new one each time | **reused** (faster) |

One request: both work. Several requests to the same site: use a **session**.

In [4]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/157.0.0.0 Safari/537.36",   # a real Chrome User-Agent
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()
session.headers.update(HEADERS)   # set ONCE: sent with every request of this session

In [ ]:
response = session.get(WIKIPEDIA_URL, timeout=20)   # no headers= needed: the session has them

In [6]:
response.raise_for_status()
print("HTTP status:", response.status_code)

HTTP status: 200


In [7]:
soup = BeautifulSoup(response.text, "html.parser")
print("Downloaded characters:", len(response.text))

Downloaded characters: 571375


### Reuse the session for a second page

Same session: same User-Agent (no headers= to repeat), same open connection, and the cookies if the site sets some.

In [ ]:
import time

time.sleep(2)   # be polite: a pause between two pages
other = session.get("https://en.wikipedia.org/wiki/S%26P_500", timeout=20)

print("HTTP status:", other.status_code)
print("User-Agent sent:", other.request.headers["User-Agent"])
print("Cookies kept by the session:", len(session.cookies))

## 2. Inspect one tag

A **tag** is one HTML element. Inspect the document title before targeting table markup.

In [8]:
title_tag = soup.title
title_tag

<title>List of S&amp;P 500 companies - Wikipedia</title>

## 3. Use `soup.find`

`find` returns the first matching tag, or `None` if there is no match.

In [9]:
first_heading = soup.find("h1")
first_heading

<h1 class="firstHeading mw-first-heading" id="firstHeading"><span dir="ltr" lang="en"><span class="mw-page-title-main">List of S&amp;P 500 companies</span></span></h1>

## 4. Use `soup.find_all`

`find_all` returns every match as a list-like `ResultSet`.

In [10]:
heading_tags = soup.find_all("h1")
len(heading_tags)

1

## 5. Find `<span>` tags

Spans often hold small inline parts of a page. Start by counting them.

In [11]:
span_tags = soup.find_all("span")
print("Number of <span> tags:", len(span_tags))

Number of <span> tags: 179


In [12]:
example_span = span_tags[0] if span_tags else None
example_span

<span class="vector-icon mw-ui-icon-menu mw-ui-icon-wikimedia-menu cdx-button__icon"></span>

## 6. Read tag properties

Read the tag's name, text, complete attribute dictionary, and class safely.

In [13]:
if example_span is None:
    print("No <span> tag was returned.")
else:
    print("name :", example_span.name)
    print("text :", example_span.get_text(" ", strip=True))
    print("attrs :", example_span.attrs)
    print("class:", example_span.get("class"))

name : span
text : 
attrs : {'class': ['vector-icon', 'mw-ui-icon-menu', 'mw-ui-icon-wikimedia-menu', 'cdx-button__icon']}
class: ['vector-icon', 'mw-ui-icon-menu', 'mw-ui-icon-wikimedia-menu', 'cdx-button__icon']


## 6a. Compare `contents` and text

`contents` keeps a tag's immediate child nodes; `get_text` combines the readable text.

In [14]:
(title_tag.contents, title_tag.get_text(" ", strip=True)) if title_tag else ([], None)

(['List of S&P 500 companies - Wikipedia'],
 'List of S&P 500 companies - Wikipedia')

## 6b. Move to a parent tag

The parsed page is a tree. A tag's `parent` is the element that contains it.

In [15]:
first_heading.parent.name if first_heading and first_heading.parent else None

'header'

## 6c. Extract a link safely

Find one anchor tag first. Its `href` may be absent, so use `.get` rather than attribute indexing.

In [16]:
first_link = soup.find("a")
first_link

<a class="mw-jump-link" href="#bodyContent">Jump to content</a>

## 6d. Read the link URL

This returns the URL or `None` instead of raising an error when there is no link or no `href`.

In [17]:
first_link.get("href") if first_link else None

'#bodyContent'

## 6e. Match an attribute with `find`

Pass an `attrs` dictionary when a meaningful HTML attribute identifies the tag.

In [18]:
soup.find("table", attrs={"id": "constituents"})

<table class="wikitable sortable mw-collapsible sticky-header" id="constituents">
<tbody id="mwIg"><tr id="mwIw">
<th id="mwJA"><a href="https://en.wikipedia.org/wiki/Ticker_symbol" id="mwJQ" rel="mw:WikiLink" title="Ticker symbol">Symbol</a></th>
<th id="mwJg">Security</th><th id="mwJw"><a href="https://en.wikipedia.org/wiki/Global_Industry_Classification_Standard" id="mwKA" rel="mw:WikiLink" title="Global Industry Classification Standard">GICS</a> Sector</th><th id="mwKQ">GICS Sub-Industry</th><th id="mwKg">Headquarters Location</th><th id="mwKw">Date added</th><th id="mwLA"><a href="https://en.wikipedia.org/wiki/Central_Index_Key" id="mwLQ" rel="mw:WikiLink" title="Central Index Key">CIK</a></th><th id="mwLg">Founded</th></tr>
<tr id="mwLw">
<td id="mwMA"><a about="#mwt11" class="external text" data-mw='{"parts":[{"template":{"target":{"wt":"NyseSymbol","href":"./Template:NyseSymbol"},"params":{"1":{"wt":"MMM"}},"i":0}}]}' href="https://www.nyse.com/quote/XNYS:MMM" id="mwMQ" rel="mw

## 6f. Match classes with `class_`

Because `class` is a Python keyword, Beautiful Soup uses `class_` in `find` and `find_all`.

In [19]:
soup.find_all("table", class_="wikitable")

[<table class="wikitable sortable mw-collapsible sticky-header" id="constituents">
 <tbody id="mwIg"><tr id="mwIw">
 <th id="mwJA"><a href="https://en.wikipedia.org/wiki/Ticker_symbol" id="mwJQ" rel="mw:WikiLink" title="Ticker symbol">Symbol</a></th>
 <th id="mwJg">Security</th><th id="mwJw"><a href="https://en.wikipedia.org/wiki/Global_Industry_Classification_Standard" id="mwKA" rel="mw:WikiLink" title="Global Industry Classification Standard">GICS</a> Sector</th><th id="mwKQ">GICS Sub-Industry</th><th id="mwKg">Headquarters Location</th><th id="mwKw">Date added</th><th id="mwLA"><a href="https://en.wikipedia.org/wiki/Central_Index_Key" id="mwLQ" rel="mw:WikiLink" title="Central Index Key">CIK</a></th><th id="mwLg">Founded</th></tr>
 <tr id="mwLw">
 <td id="mwMA"><a about="#mwt11" class="external text" data-mw='{"parts":[{"template":{"target":{"wt":"NyseSymbol","href":"./Template:NyseSymbol"},"params":{"1":{"wt":"MMM"}},"i":0}}]}' href="https://www.nyse.com/quote/XNYS:MMM" id="mwMQ" r

## 6g. Filter by an exact class

A tag's classes are stored as a list. Equality checks the exact class list while filtering an existing `find_all` result.

In [20]:
exact_class_spans = [tag for tag in span_tags if example_span and tag.get("class") == example_span.get("class")]
len(exact_class_spans)

1

## 6h. Use CSS selectors

`select_one` is like `find`, and `select` is like `find_all`, but both use CSS-selector syntax.

In [21]:
soup.select("table.wikitable")

[<table class="wikitable sortable mw-collapsible sticky-header" id="constituents">
 <tbody id="mwIg"><tr id="mwIw">
 <th id="mwJA"><a href="https://en.wikipedia.org/wiki/Ticker_symbol" id="mwJQ" rel="mw:WikiLink" title="Ticker symbol">Symbol</a></th>
 <th id="mwJg">Security</th><th id="mwJw"><a href="https://en.wikipedia.org/wiki/Global_Industry_Classification_Standard" id="mwKA" rel="mw:WikiLink" title="Global Industry Classification Standard">GICS</a> Sector</th><th id="mwKQ">GICS Sub-Industry</th><th id="mwKg">Headquarters Location</th><th id="mwKw">Date added</th><th id="mwLA"><a href="https://en.wikipedia.org/wiki/Central_Index_Key" id="mwLQ" rel="mw:WikiLink" title="Central Index Key">CIK</a></th><th id="mwLg">Founded</th></tr>
 <tr id="mwLw">
 <td id="mwMA"><a about="#mwt11" class="external text" data-mw='{"parts":[{"template":{"target":{"wt":"NyseSymbol","href":"./Template:NyseSymbol"},"params":{"1":{"wt":"MMM"}},"i":0}}]}' href="https://www.nyse.com/quote/XNYS:MMM" id="mwMQ" r

## 7. Filter an existing `find_all` result to an exact tag

First gather every table. Then filter that existing list by the table's `id` attribute. At the time of writing, the constituent-company table is `id="constituents"`.

In [22]:
table_tags = soup.find_all("table")
print("Number of <table> tags:", len(table_tags))

Number of <table> tags: 2


In [23]:
table = next((tag for tag in table_tags if tag.get("id") == "constituents"), None)
if table is None:
    raise RuntimeError("Could not find table#constituents. Inspect the current page HTML and update the lesson selector.")
table

<table class="wikitable sortable mw-collapsible sticky-header" id="constituents">
<tbody id="mwIg"><tr id="mwIw">
<th id="mwJA"><a href="https://en.wikipedia.org/wiki/Ticker_symbol" id="mwJQ" rel="mw:WikiLink" title="Ticker symbol">Symbol</a></th>
<th id="mwJg">Security</th><th id="mwJw"><a href="https://en.wikipedia.org/wiki/Global_Industry_Classification_Standard" id="mwKA" rel="mw:WikiLink" title="Global Industry Classification Standard">GICS</a> Sector</th><th id="mwKQ">GICS Sub-Industry</th><th id="mwKg">Headquarters Location</th><th id="mwKw">Date added</th><th id="mwLA"><a href="https://en.wikipedia.org/wiki/Central_Index_Key" id="mwLQ" rel="mw:WikiLink" title="Central Index Key">CIK</a></th><th id="mwLg">Founded</th></tr>
<tr id="mwLw">
<td id="mwMA"><a about="#mwt11" class="external text" data-mw='{"parts":[{"template":{"target":{"wt":"NyseSymbol","href":"./Template:NyseSymbol"},"params":{"1":{"wt":"MMM"}},"i":0}}]}' href="https://www.nyse.com/quote/XNYS:MMM" id="mwMQ" rel="mw

## 7a. Inspect a table row

Tables contain rows (`tr`). Select the first row after the header before reading its cells.

In [24]:
first_body_row = table.find("tr").find_next_sibling("tr") if table and table.find("tr") else None

## 7b. Inspect a row's cells

A data row contains `td` cells. `recursive=False` keeps the search to that row's direct cells.

In [25]:
first_body_row.find_all("td", recursive=False) if first_body_row else []

[<td id="mwMA"><a about="#mwt11" class="external text" data-mw='{"parts":[{"template":{"target":{"wt":"NyseSymbol","href":"./Template:NyseSymbol"},"params":{"1":{"wt":"MMM"}},"i":0}}]}' href="https://www.nyse.com/quote/XNYS:MMM" id="mwMQ" rel="mw:ExtLink nofollow" typeof="mw:Transclusion">MMM</a></td>,
 <td id="mwMg"><a href="https://en.wikipedia.org/wiki/3M" id="mwMw" rel="mw:WikiLink" title="3M">3M</a></td>,
 <td id="mwNA">Industrials</td>,
 <td id="mwNQ">Industrial Conglomerates</td>,
 <td id="mwNg"><a href="https://en.wikipedia.org/wiki/Saint_Paul,_Minnesota" id="mwNw" rel="mw:WikiLink" title="Saint Paul, Minnesota">Saint Paul, Minnesota</a></td>,
 <td id="mwOA">1957-03-04</td>,
 <td id="mwOQ">0000066740</td>,
 <td id="mwOg">1902</td>]

## 8. Introduce reusable logic

A small helper keeps table-cell text extraction consistent.

In [26]:
def text_or_none(tag):
    return tag.get_text(" ", strip=True) if tag else None

## 9. Introduce CSS selectors

`select_one` finds the first CSS-selector match. This is the CSS form of the exact table lookup.

In [27]:
table_css = soup.select_one("table#constituents")
print(text_or_none(table_css.find("caption")) if table_css else "Table not found")

None


## 10. Extract table headers

Now apply `find` and `find_all` inside the exact table.

In [28]:
header_row = table.find("tr")
if header_row is None:
    raise RuntimeError("The constituents table has no header row.")

In [29]:
header_cells = header_row.find_all("th")
headers = [text_or_none(cell) for cell in header_cells]
headers

['Symbol',
 'Security',
 'GICS Sector',
 'GICS Sub-Industry',
 'Headquarters Location',
 'Date added',
 'CIK',
 'Founded']

## 11. Extract records, then make a pandas table

Keep only body rows with the same number of cells as headers so notes or malformed rows do not become records.

In [30]:
body_rows = table.find_all("tr")[1:]

In [31]:
rows = []
for row in body_rows:
    cells = row.find_all("td")
    if len(cells) == len(headers):
        rows.append([text_or_none(cell) for cell in cells])

In [32]:
sp500 = pd.DataFrame(rows, columns=headers)
print(f"Extracted {len(sp500)} company rows and {len(sp500.columns)} columns.")

Extracted 503 company rows and 8 columns.


In [33]:
sp500.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee , Wisconsin",2017-07-26,0000091142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin , Ireland",2011-07-06,0001467373,1989


## 12. Make symbols Yahoo-compatible

Keep Wikipedia's original symbol and create a separate Yahoo-compatible version. For example, Yahoo uses `BRK-B` where the table can show `BRK.B`.

In [34]:
if "Symbol" not in sp500.columns:
    raise KeyError(f"Expected a Symbol column; found: {sp500.columns.tolist()}")

In [35]:
sp500["Yahoo symbol"] = sp500["Symbol"].str.replace(".", "-", regex=False)
sp500[["Symbol", "Yahoo symbol", "Security", "GICS Sector"]].head(10)

,Symbol,Yahoo symbol,Security,GICS Sector
0,MMM,MMM,3M,Industrials
1,AOS,AOS,A. O. Smith,Industrials
2,ABT,ABT,Abbott Laboratories,Health Care
3,ABBV,ABBV,AbbVie,Health Care
4,ACN,ACN,Accenture,Information Technology
5,ADBE,ADBE,Adobe Inc.,Information Technology
6,AMD,AMD,Advanced Micro Devices,Information Technology
7,AES,AES,AES Corporation,Utilities
8,AFL,AFL,Aflac,Financials
9,A,A,Agilent Technologies,Health Care


## Checkpoint

You practiced **fetch → parse → inspect a tag → `find` → `find_all` → spans → tag properties → filter results → reusable logic → CSS selectors → pandas**. The third notebook can use this table's first two Yahoo-compatible symbols for exactly two Yahoo Finance requests.